In [18]:
import numpy as np
import pandas as pd
import  tpqoa
import time
from datetime import datetime, timedelta, timezone
import warnings
warnings.filterwarnings('ignore')

In [20]:
class ConTrader(tpqoa.tpqoa):
    def __init__(self, conf_file, instrument, bar_length, window, units):
        super().__init__(conf_file)
        self.instrument = instrument
        self.bar_length = pd.to_timedelta(bar_length)
        self.tick_data = pd.DataFrame()
        self.raw_data = None
        self.data = None
        self.last_bar = None
        self.units = units
        self.position = 0
        self.profits = [] # NEW

        #*****************add strategy-specific attributes here******************
        self.window = window
        #************************************************************************

    def get_most_recent(self, days = 5):
        while True:
            time.sleep(2)
            now = datetime.now(timezone.utc).replace(tzinfo=None)
            now = now - timedelta(microseconds = now.microsecond)
            past = now - timedelta(days = days)
            df = self.get_history(instrument = self.instrument, start = past, end = now,
                                   granularity = "S5", price = "M", localize = False).c.dropna().to_frame()
            df.rename(columns = {"c":self.instrument}, inplace = True)
            df = df.resample(self.bar_length, label = "right").last().dropna().iloc[:-1]
            self.raw_data = df.copy()
            self.last_bar = self.raw_data.index[-1]
            if pd.to_datetime(datetime.now(timezone.utc)) - self.last_bar < self.bar_length:
                break

    def on_success(self, time, bid, ask):
        print(self.ticks, end = " ")

        # collect and store tick data
        recent_tick = pd.to_datetime(time)
        df = pd.DataFrame({self.instrument:(ask + bid)/2},
                          index = [recent_tick])
        self.tick_data = pd.concat([self.tick_data, df]) # new with pd.concat()

        # if a time longer than the bar_lenght has elapsed between last full bar and the most recent tick
        if recent_tick - self.last_bar >= self.bar_length:
            self.resample_and_join()
            self.define_strategy()
            self.execute_trades()

    def resample_and_join(self):
        self.raw_data = pd.concat([self.raw_data, self.tick_data.resample(self.bar_length,
                                                                          label="right").last().ffill().iloc[:-1]])
        self.tick_data = self.tick_data.iloc[-1:]
        self.last_bar = self.raw_data.index[-1]

    def define_strategy(self): # "strategy-specific"
        df = self.raw_data.copy()

        #******************** define your strategy here ************************
        df["returns"] = np.log(df[self.instrument] / df[self.instrument].shift())
        df["position"] = -np.sign(df.returns.rolling(self.window).mean())
        #***********************************************************************

        self.data = df.copy()

    def execute_trades(self):
        if self.data["position"].iloc[-1] == 1:
            if self.position == 0:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")  # NEW
            elif self.position == -1:
                order = self.create_order(self.instrument, self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")  # NEW
            self.position = 1
        elif self.data["position"].iloc[-1] == -1:
            if self.position == 0:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")  # NEW
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")  # NEW
            self.position = -1
        elif self.data["position"].iloc[-1] == 0:
            if self.position == -1:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")  # NEW
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")  # NEW
            self.position = 0

    def report_trade(self, order, going):  # NEW
        time = order["time"]
        units = order["units"]
        price = order["price"]
        pl = float(order["pl"])
        self.profits.append(pl)
        cumpl = sum(self.profits)
        print("\n" + 100* "-")
        print("{} | {}".format(time, going))
        print("{} | units = {} | price = {} | P&L = {} | Cum P&L = {}".format(time, units, price, pl, cumpl))
        print(100 * "-" + "\n")


In [24]:
trader = ConTrader('oanda.cfg', 'EUR_USD', bar_length='1min', window=60, units=1000000)

In [25]:
trader.get_most_recent()
trader.stream_data(trader.instrument, stop= 200)

if trader.position!= 1:
    close_order= trader.create_order(trader.instrument, units=-trader.position * trader.units, suppress=True, ret=True)
    trader.report_trade(close_order, "GOING NEUTRAL")
    trader.position=0

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 
----------------------------------------------------------------------------------------------------
2026-02-11T12:16:00.711508458Z | GOING LONG
2026-02-11T12:16:00.711508458Z | units = 1000000.0 | price = 1.19073 | P&L = 0.0 | Cum P&L = 0.0
----------------------------------------------------------------------------------------------------

16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191

In [26]:
import numpy as np
import pandas as pd
import  tpqoa

In [ ]:
api = tpqoa.tpqoa('oanda.cfg')

In [27]:
api.stream_data('EUR_USD', stop=10)

2026-02-11T12:19:45.737438130Z 1.19052 1.19062
2026-02-11T12:19:48.387382189Z 1.19053 1.19063
2026-02-11T12:19:48.663411832Z 1.19055 1.19065
2026-02-11T12:19:49.277221685Z 1.19056 1.19065
2026-02-11T12:19:49.509699334Z 1.19055 1.19065
2026-02-11T12:19:51.017365840Z 1.19056 1.19065
2026-02-11T12:19:51.456456030Z 1.19055 1.19065
2026-02-11T12:19:53.295886961Z 1.19055 1.19065
2026-02-11T12:19:59.737265030Z 1.19056 1.19065
2026-02-11T12:20:00.063511853Z 1.19055 1.19065


In [29]:
api.on_success('2026-02-11T12:20:00', 1.19055, 1.19065)

2026-02-11T12:20:00 1.19055 1.19065


In [41]:
class Clone(tpqoa.tpqoa):
    def __init__(self, config_file):
        super().__init__(config_file)
        self.tick_data = pd.DataFrame()

    def on_success(self, time, bid, ask):
        print(f"date: {time} | bid: {bid} | ask: {ask}")
        df = pd.DataFrame({"bid":bid, "ask":ask},
                          index=[pd.to_datetime(time)])
        self.tick_data = pd.concat([self.tick_data, df])

In [42]:
api = Clone("oanda.cfg")

In [49]:
api.stream_data('EUR_USD', stop=20)

date: 2026-02-11T12:44:30.095744164Z | bid: 1.19013 | ask: 1.19022
date: 2026-02-11T12:44:33.268798203Z | bid: 1.19015 | ask: 1.19025
date: 2026-02-11T12:44:36.763680832Z | bid: 1.19017 | ask: 1.19027
date: 2026-02-11T12:44:37.065396036Z | bid: 1.19015 | ask: 1.19025
date: 2026-02-11T12:44:38.203968303Z | bid: 1.19015 | ask: 1.19026
date: 2026-02-11T12:44:38.429096401Z | bid: 1.19016 | ask: 1.19025
date: 2026-02-11T12:44:39.320947604Z | bid: 1.19016 | ask: 1.19024
date: 2026-02-11T12:44:40.041473701Z | bid: 1.19016 | ask: 1.19025
date: 2026-02-11T12:44:48.259612746Z | bid: 1.19016 | ask: 1.19026
date: 2026-02-11T12:44:48.681018062Z | bid: 1.19017 | ask: 1.19025
date: 2026-02-11T12:44:48.890426826Z | bid: 1.19016 | ask: 1.19026
date: 2026-02-11T12:44:50.898187850Z | bid: 1.19015 | ask: 1.19024
date: 2026-02-11T12:44:51.346248048Z | bid: 1.19015 | ask: 1.19027
date: 2026-02-11T12:44:52.097428616Z | bid: 1.19015 | ask: 1.19025
date: 2026-02-11T12:44:52.823144570Z | bid: 1.19015 | ask: 1.1

In [50]:
api.tick_data

,bid,ask
2026-02-11 12:37:58.156878839+00:00,1.19014,1.19024
2026-02-11 12:37:58.372160003+00:00,1.19015,1.19024
2026-02-11 12:37:58.567256699+00:00,1.19013,1.19023
2026-02-11 12:38:00.586739087+00:00,1.19011,1.19021
2026-02-11 12:39:43.513152798+00:00,1.19015,1.19023
2026-02-11 12:39:49.847129946+00:00,1.19014,1.19022
2026-02-11 12:39:50.478678054+00:00,1.19015,1.19023
2026-02-11 12:39:50.927015989+00:00,1.19014,1.19022
2026-02-11 12:44:30.095744164+00:00,1.19013,1.19022
2026-02-11 12:44:33.268798203+00:00,1.19015,1.19025


In [52]:
api.tick_data.resample('10s', label='right').last().dropna()

,bid,ask
2026-02-11 12:38:00+00:00,1.19013,1.19023
2026-02-11 12:38:10+00:00,1.19011,1.19021
2026-02-11 12:39:50+00:00,1.19014,1.19022
2026-02-11 12:40:00+00:00,1.19014,1.19022
2026-02-11 12:44:40+00:00,1.19016,1.19024
2026-02-11 12:44:50+00:00,1.19016,1.19026
2026-02-11 12:45:00+00:00,1.19022,1.19032


In [ ]:
2